# 🔬 TikTok Acne Misinformation — Full Data Pipeline

**What this notebook does:**
1. Mount Google Drive (single account)
2. Install dependencies (pinned versions)
3. Configure paths + API key
4. Load CSV
5. Scrape captions + metadata via yt-dlp (no download)
6. Download videos to Google Drive at 360p
7. Transcribe with Whisper large-v3 on GPU
8. Classify all 23 Group 2 fields with Claude API
9. Validate + summary stats
10. Export final CSV

> ⚙️ **Before running:** Runtime → Change runtime type → **T4 GPU**
>
> 📁 **Upload** `Final_Features_Parth__Final__data_set.csv` to `MyDrive/TikTok_Acne_Research/data/` in your Drive before running Cell 4.
>
> 🔑 **Paste your Anthropic API key** in Cell 3.

In [ ]:
# ── STEP 1: Mount Google Drive ──────────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR    = '/content/drive/MyDrive/TikTok_Acne_Research'
DATA_DIR    = f'{BASE_DIR}/data'
VIDEO_DIR   = f'{BASE_DIR}/videos'

os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)

print('✅ Drive mounted')
print(f'   Data   → {DATA_DIR}')
print(f'   Videos → {VIDEO_DIR}')

In [ ]:
# ── STEP 2: Install dependencies (pinned versions) ──────────────────
# faster-whisper pinned to avoid ctranslate2/CUDA mismatch on T4
!pip install -q yt-dlp anthropic pandas tqdm
!pip install -q faster-whisper==1.0.3 ctranslate2==4.4.0
!apt-get install -qq ffmpeg
print('✅ All dependencies installed')

In [ ]:
# ── STEP 3: Configuration ───────────────────────────────────────────

# 🔑 Paste your Anthropic API key here
ANTHROPIC_API_KEY = 'sk-ant-YOUR_KEY_HERE'

# Pipeline settings
SKIP_EXISTING = True   # Skip videos already downloaded to Drive

# File paths (all saved to your Google Drive)
CSV_INPUT       = f'{DATA_DIR}/Final_Features_Parth__Final__data_set.csv'
CSV_META        = f'{DATA_DIR}/step1_metadata.csv'
CSV_TRANSCRIBED = f'{DATA_DIR}/step2_transcribed.csv'
CSV_FINAL       = f'{DATA_DIR}/step3_classified_final.csv'

# Group 2 fields to classify
GROUP2_COLS = [
    'creator_type', 'creator_motivation', 'subject_of_experience',
    'skin_condition_type', 'treatment_type', 'outcome_type',
    'side_effects_mentioned', 'video_intent', 'intent_clarity',
    'product_name', 'product_brand', 'sponsored_content',
    'affiliate_link_present', 'promotion_style', 'before_after_claim',
    'time_to_result_claim', 'unrealistic_claim_flag', 'Missed_info',
    'misinformation_label', 'claim_type', 'claim_accuracy',
    'evidence_present', 'label_confidence'
]

print('✅ Config set')
print(f'   API key: {"SET ✅" if ANTHROPIC_API_KEY != "sk-ant-YOUR_KEY_HERE" else "❌ MISSING — paste your key above"}')
print(f'   Group 2 fields to fill: {len(GROUP2_COLS)}')

In [ ]:
# ── STEP 4: Load CSV ────────────────────────────────────────────────
import pandas as pd, os

if not os.path.exists(CSV_INPUT):
    from google.colab import files
    print('CSV not found in Drive — upload it now...')
    uploaded = files.upload()
    import shutil
    shutil.copy(list(uploaded.keys())[0], CSV_INPUT)

df = pd.read_csv(CSV_INPUT)

# Always extract from URL — avoids NaN video_id bug
df['creator_username'] = df['video_url'].str.extract(r'@([^/]+)')
df['video_id']         = df['video_url'].str.extract(r'/video/(\d+)')

print(f'✅ Loaded: {len(df)} rows, {df["creator_username"].nunique()} unique creators')
df[['creator_username', 'video_id', 'video_url']].head(3)

In [ ]:
# ── STEP 5: Scrape captions + metadata via yt-dlp (no download) ─────
import subprocess, json, time, pandas as pd
from tqdm import tqdm

def scrape_metadata(url, retries=2):
    cmd = [
        'yt-dlp', '--no-download', '--dump-json', '--no-warnings',
        '--no-check-certificates',
        '--user-agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        '--add-header', 'Accept-Language:en-US,en;q=0.9',
        url
    ]
    for attempt in range(retries + 1):
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
            if r.returncode == 0 and r.stdout.strip():
                d = json.loads(r.stdout.strip())
                return {
                    'caption':          d.get('description', '') or d.get('title', ''),
                    'hashtags':         ' '.join(d.get('tags', [])),
                    'creator_bio':      d.get('uploader', ''),
                    'creator_username_scraped': d.get('uploader_id', '').lstrip('@'),
                    'creator_id':       d.get('channel_id', ''),
                    'views':            d.get('view_count', 0),
                    'likes':            d.get('like_count', 0),
                    'comments_count':   d.get('comment_count', 0),
                    'shares':           d.get('repost_count', 0),
                    'video_duration_seconds': d.get('duration', 0),
                    'posted_date':      d.get('upload_date', ''),
                    'Language':         d.get('language', ''),
                    'scrape_status':    'success'
                }
        except Exception:
            if attempt < retries:
                time.sleep(2 ** attempt)
    return {'scrape_status': 'failed', 'caption': '', 'creator_bio': '',
            'creator_username_scraped': ''}

# Resume from checkpoint if exists
ckpt = CSV_META + '.checkpoint'
if os.path.exists(ckpt):
    done_df  = pd.read_csv(ckpt)
    done_urls = set(done_df['video_url'].tolist())
    meta_results = done_df.to_dict('records')
    print(f'Resuming — {len(done_urls)} already scraped')
else:
    done_urls, meta_results = set(), []

urls = df['video_url'].tolist()
remaining = [u for u in urls if u not in done_urls]
print(f'Scraping {len(remaining)} remaining URLs...\n')

for i, url in enumerate(tqdm(remaining)):
    result = scrape_metadata(url)
    result['video_url'] = url
    meta_results.append(result)
    time.sleep(0.5)
    if (i + 1) % 50 == 0:
        pd.DataFrame(meta_results).to_csv(ckpt, index=False)
        ok = sum(1 for r in meta_results if r.get('scrape_status') == 'success')
        print(f'  Checkpoint {len(meta_results)}: {ok} succeeded')

meta_df = pd.DataFrame(meta_results)

# Only drop cols that already exist AND are non-empty in df (keep scraped values)
overlap_cols = [
    c for c in meta_df.columns
    if c in df.columns and c != 'video_url' and df[c].notna().any()
]
meta_df = meta_df.drop(columns=overlap_cols)

# Use scraped username as the authoritative creator_username
if 'creator_username_scraped' in meta_df.columns:
    meta_df = meta_df.rename(columns={'creator_username_scraped': 'creator_username'})
    df = df.drop(columns=['creator_username'], errors='ignore')

df_enriched = df.merge(meta_df, on='video_url', how='left')
df_enriched.to_csv(CSV_META, index=False)

ok = (meta_df['scrape_status'] == 'success').sum()
print(f'\n✅ Scrape complete: {ok}/{len(urls)} succeeded')
print(f'   Saved → {CSV_META}')


In [ ]:
# ── STEP 6: Download videos to Google Drive at 360p ─────────────────
import subprocess, os, time, pandas as pd
from tqdm import tqdm

df_enriched = pd.read_csv(CSV_META)

def download_video(url, output_dir, skip_existing=True):
    # Fix Bug 6: always use video_id from URL, never NaN
    vid_id   = url.split('/')[-1]
    out_path = os.path.join(output_dir, f'{vid_id}.mp4')

    if skip_existing and os.path.exists(out_path) and os.path.getsize(out_path) > 50_000:
        return {'status': 'skipped', 'video_path': out_path, 'video_id': vid_id}

    cmd = [
        'yt-dlp',
        '-f', 'bestvideo[height<=360]+bestaudio/best[height<=360]/best',
        '--merge-output-format', 'mp4',
        '--no-warnings', '--no-check-certificates',
        '--user-agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        '--add-header', 'Accept-Language:en-US,en;q=0.9',
        '-o', out_path, url
    ]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        if r.returncode == 0 and os.path.exists(out_path):
            return {'status': 'success', 'video_path': out_path,
                    'video_id': vid_id,
                    'size_mb': round(os.path.getsize(out_path) / 1024 / 1024, 1)}
        return {'status': 'failed', 'video_path': None, 'video_id': vid_id,
                'error': r.stderr[-300:]}
    except subprocess.TimeoutExpired:
        return {'status': 'timeout', 'video_path': None, 'video_id': vid_id}
    except Exception as e:
        return {'status': 'error', 'video_path': None, 'video_id': vid_id, 'error': str(e)}

results, total_mb = [], 0
print(f'Downloading {len(df_enriched)} videos to Drive at 360p...')
print(f'Estimated total: ~3GB\n')

for _, row in tqdm(df_enriched.iterrows(), total=len(df_enriched)):
    vid_id = row['video_url'].split('/')[-1]  # always from URL
    res    = download_video(row['video_url'], VIDEO_DIR, SKIP_EXISTING)
    res['video_url'] = row['video_url']
    results.append(res)
    if res.get('size_mb'): total_mb += res['size_mb']

    if len(results) % 50 == 0:
        done   = sum(1 for r in results if r['status'] in ('success', 'skipped'))
        failed = sum(1 for r in results if r['status'] not in ('success', 'skipped'))
        print(f'  {len(results)}: ✅ {done} | ❌ {failed} | 💾 {total_mb:.0f}MB')
    time.sleep(0.3)

dl_df = pd.DataFrame(results)

# Fix Bug 3: correct pandas mapping (no .get() on DataFrame)
path_map = dl_df.set_index('video_url')['video_path']
stat_map = dl_df.set_index('video_url')['status']
df_enriched['video_path'] = df_enriched['video_url'].map(path_map)
df_enriched['dl_status']  = df_enriched['video_url'].map(stat_map)
df_enriched.to_csv(CSV_META, index=False)

ok = dl_df['status'].isin(['success', 'skipped']).sum()
print(f'\n✅ Downloads complete: {ok}/{len(df_enriched)}')
print(f'   Total size: {total_mb:.0f}MB ({total_mb/1024:.1f}GB)')
print(f'   Saved → {VIDEO_DIR}')


In [ ]:
# ── STEP 7: Transcribe with Whisper large-v3 (GPU) ──────────────────
from faster_whisper import WhisperModel
import os, pandas as pd
from tqdm import tqdm

print('Loading Whisper large-v3 on GPU...')
model = WhisperModel('large-v3', device='cuda', compute_type='float16')
print('✅ Model loaded\n')

df_enriched = pd.read_csv(CSV_META)

def transcribe(video_path):
    if not video_path or not os.path.exists(str(video_path)):
        return '', 'unknown'
    try:
        segs, info = model.transcribe(
            str(video_path), beam_size=5, language=None, vad_filter=True
        )
        return ' '.join(s.text.strip() for s in segs).strip(), info.language
    except Exception:
        return '', 'error'

# Resume from checkpoint — only load completed rows (non-NaN transcripts)
ckpt = CSV_TRANSCRIBED + '.checkpoint'
if os.path.exists(ckpt):
    ckpt_df     = pd.read_csv(ckpt)
    # Only keep rows that were actually transcribed (not NaN padding)
    done_rows   = ckpt_df[ckpt_df['transcript'].notna() & (ckpt_df['transcript'] != '')]
    transcripts = done_rows['transcript'].tolist()
    languages   = done_rows['Language_whisper'].tolist()
    print(f'Resuming from checkpoint: {len(transcripts)} already transcribed')
else:
    transcripts, languages = [], []

start_idx = len(transcripts)
remaining = df_enriched.iloc[start_idx:]
print(f'Transcribing {len(remaining)} remaining videos...')
print(f'Estimated time: ~2-4 hours on T4 GPU\n')

for i, (_, row) in enumerate(tqdm(remaining.iterrows(), total=len(remaining))):
    text, lang = transcribe(row.get('video_path'))
    transcripts.append(text)
    languages.append(lang)

    # Fix Bug 7: use iloc (position-based), not loc (label-based)
    if (start_idx + i + 1) % 50 == 0:
        pad_t = max(0, len(df_enriched) - len(transcripts))
        pad_l = max(0, len(df_enriched) - len(languages))
        df_enriched['transcript']       = transcripts + [''] * pad_t
        df_enriched['Language_whisper'] = languages   + [''] * pad_l
        df_enriched.to_csv(ckpt, index=False)
        done = sum(1 for t in transcripts if t)
        print(f'  {len(transcripts)}: {done} transcribed')

df_enriched['transcript']       = transcripts
df_enriched['Language_whisper'] = languages
df_enriched.to_csv(CSV_TRANSCRIBED, index=False)

done = sum(1 for t in transcripts if t)
print(f'\n✅ Transcription complete: {done}/{len(df_enriched)}')
print(f'   Saved → {CSV_TRANSCRIBED}')


In [ ]:
# ── STEP 8: Classify all 23 Group 2 fields with Claude API ──────────
import anthropic, json, time, os, pandas as pd
from tqdm import tqdm

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """You are a dermatology misinformation researcher classifying TikTok acne videos.
Given a video's caption, audio transcript, creator username, bio, and hashtags — classify all fields.
Respond ONLY with a valid JSON object. No markdown, no explanation, no preamble.

TAXONOMY (use only these values):
creator_type: dermatologist | influencer | patient | brand | other
creator_motivation: educational | promotional | personal_experience | entertainment | mixed
subject_of_experience: self | patient | general_advice | unknown
skin_condition_type: acne | cystic_acne | hormonal_acne | blackheads | whiteheads | acne_scarring | general_skin | unknown
treatment_type: topical | oral_medication | dietary | supplement | device | lifestyle | procedure | natural_remedy | none
outcome_type: cure | improvement | prevention | management | none
side_effects_mentioned: yes | no
video_intent: inform | sell | entertain | inspire | warn | none
intent_clarity: clear | somewhat_clear | unclear
product_name: exact product name as string, or null
product_brand: exact brand name as string, or null
sponsored_content: yes | no | unclear
affiliate_link_present: yes | no | unclear
promotion_style: hard_sell | soft_sell | educational | testimonial | none
before_after_claim: yes | no
time_to_result_claim: days | weeks | months | none
unrealistic_claim_flag: yes | no
Missed_info: brief string describing key omitted clinical info, or null
misinformation_label: yes | no | partial
claim_type: dietary_cure | diy_topicals | supplements | gut_health | led_device | anti_isotretinoin | hormone_supplement | ice_therapy | sunscreen_avoidance | clinically_accurate | product_promotion | general_advice | none
claim_accuracy: supported | partial | unsupported | false | na
evidence_present: yes | no | anecdotal
label_confidence: high | medium | low"""

DEFAULT_FIELDS = {
    'creator_type': 'unknown', 'creator_motivation': 'unknown',
    'subject_of_experience': 'unknown', 'skin_condition_type': 'unknown',
    'treatment_type': 'none', 'outcome_type': 'none',
    'side_effects_mentioned': 'no', 'video_intent': 'none',
    'intent_clarity': 'unclear', 'product_name': None,
    'product_brand': None, 'sponsored_content': 'unclear',
    'affiliate_link_present': 'unclear', 'promotion_style': 'none',
    'before_after_claim': 'no', 'time_to_result_claim': 'none',
    'unrealistic_claim_flag': 'no', 'Missed_info': None,
    'misinformation_label': 'no', 'claim_type': 'none',
    'claim_accuracy': 'na', 'evidence_present': 'no',
    'label_confidence': 'low'
}

def build_prompt(row):
    return f"""CREATOR USERNAME: {str(row.get('creator_username', '') or '')}
BIO: {str(row.get('creator_bio', '') or '')[:300]}
CAPTION: {str(row.get('caption', '') or '')[:500]}
HASHTAGS: {str(row.get('hashtags', '') or '')}
TRANSCRIPT: {str(row.get('transcript', '') or '')[:1500]}

Return a JSON object classifying this TikTok video across all 23 fields."""

def classify_row(row, retries=2):
    for attempt in range(retries + 1):
        try:
            resp = client.messages.create(
                model='claude-sonnet-4-6',
                max_tokens=700,
                system=SYSTEM_PROMPT,
                messages=[{'role': 'user', 'content': build_prompt(row)}]
            )
            text = resp.content[0].text.strip()
            if text.startswith('```'):
                text = text.split('```')[1]
                if text.startswith('json'): text = text[4:]
            return {**DEFAULT_FIELDS, **json.loads(text)}
        except json.JSONDecodeError:
            if attempt < retries: time.sleep(2)
        except anthropic.RateLimitError:
            print('Rate limit — waiting 60s...')
            time.sleep(60)
        except Exception:
            if attempt < retries: time.sleep(3)
    return DEFAULT_FIELDS.copy()

df_t = pd.read_csv(CSV_TRANSCRIBED)

# Resume from checkpoint
ckpt_path = CSV_FINAL + '.checkpoint'
if os.path.exists(ckpt_path):
    df_done   = pd.read_csv(ckpt_path)
    done_urls = set(df_done['video_url'].tolist())
    print(f'Resuming — {len(done_urls)} already classified')
else:
    df_done, done_urls = pd.DataFrame(), set()

results = []
todo    = df_t[~df_t['video_url'].isin(done_urls)]
print(f'Classifying {len(todo)} videos...')
print(f'Estimated API cost: ~${len(todo) * 0.002:.2f}\n')

for _, row in tqdm(todo.iterrows(), total=len(todo)):
    result = {'video_url': row['video_url']}
    result.update(classify_row(row))
    results.append(result)
    time.sleep(0.3)

    if len(results) % 50 == 0:
        ckpt_df = pd.concat([df_done, pd.DataFrame(results)], ignore_index=True)
        ckpt_df.to_csv(ckpt_path, index=False)
        print(f'  Checkpoint: {len(ckpt_df)} classified')

all_results = pd.concat([df_done, pd.DataFrame(results)], ignore_index=True)
df_final    = df_t.drop(columns=[c for c in GROUP2_COLS if c in df_t.columns], errors='ignore')
df_final    = df_final.merge(all_results[['video_url'] + GROUP2_COLS], on='video_url', how='left')
df_final.to_csv(CSV_FINAL, index=False)

print(f'\n✅ Classification complete: {len(all_results)}/{len(df_t)}')
print(f'   Saved → {CSV_FINAL}')


In [ ]:
# ── STEP 9: Validate results + summary stats ────────────────────────
import pandas as pd

df_final = pd.read_csv(CSV_FINAL)

# GROUP2_COLS is defined in Cell 3 — always available
filled = df_final[GROUP2_COLS].notna().all(axis=1).sum()

print('=' * 55)
print('DATASET SUMMARY')
print('=' * 55)
print(f'Total rows:             {len(df_final)}')
print(f'All Group 2 cols filled: {filled}/{len(df_final)}')
print()

for col in ['creator_type', 'misinformation_label', 'claim_type',
            'claim_accuracy', 'before_after_claim', 'label_confidence']:
    print(f'{col.upper()}')
    print(df_final[col].value_counts().to_string())
    print()

low_conf = df_final[df_final['label_confidence'] == 'low']
print(f'LOW CONFIDENCE rows (may need manual review): {len(low_conf)}')
low_conf[['video_url', 'creator_username', 'claim_type', 'misinformation_label']].head(10)

In [ ]:
# ── STEP 10: Export final CSV ────────────────────────────────────────
from google.colab import files
import pandas as pd

df_final = pd.read_csv(CSV_FINAL)

export_cols = [
    'creator_username', 'creator_id', 'creator_gender', 'creator_age_range',
    'creator_race_visual', 'creator_type', 'creator_motivation',
    'video_id', 'video_url', 'Posting date', 'video_duration_seconds',
    'No._people_video', 'video_shot_complexity', 'indoor_outdoor',
    'video_format', 'Video_type', 'face_visibility', 'subject_of_experience',
    'skin_condition_type', 'condition_extent', 'condition_duration',
    'condition_recurrence', 'treatment_type', 'outcome_type',
    'side_effects_mentioned', 'video_intent', 'intent_clarity',
    'product_name', 'product_brand', 'sponsored_content',
    'affiliate_link_present', 'promotion_style', 'before_after_claim',
    'time_to_result_claim', 'unrealistic_claim_flag',
    'views', 'likes', 'comments_count', 'shares', 'engagement_rate',
    'comment_text', 'comment_stance', 'comment_sentiment',
    'Language', 'Missed_info', 'misinformation_label',
    'claim_type', 'claim_accuracy', 'evidence_present', 'label_confidence',
    'caption', 'transcript'
]

available = [c for c in export_cols if c in df_final.columns]
df_export = df_final[available].copy()
# Rename comments_count → comments for final schema consistency
if 'comments_count' in df_export.columns:
    df_export = df_export.rename(columns={'comments_count': 'comments'})

export_path = f'{DATA_DIR}/TikTok_Acne_Group2_Labeled.csv'
df_export.to_csv(export_path, index=False)

print(f'✅ Saved to Drive: {export_path}')
print(f'   Rows: {len(df_export)} | Columns: {len(df_export.columns)}')
print('Downloading...')
files.download(export_path)
